# Predicting a post's success relative to its author

The final solution combines two gradient boosting models with a fixed class
decision rule. On the official test set it achieves **macro-F1 0.6390, QWK 0.5741,
accuracy 0.6692**; the original baseline achieved macro-F1 0.5403. This notebook
explains the hypotheses that led to the solution, the reference used for each
comparison, and the changes that were not retained.

It reads only aggregates from `reports/summary.json`. Raw posts, personal
identifiers, and individual prediction rows are not needed here.


In [1]:
import json
from pathlib import Path
import pandas as pd

summary = json.loads(Path("reports/summary.json").read_text(encoding="utf-8"))
first = {row["experiment"]: row for row in summary["first_stage"]}
def f1(name):
    return first[name]["macro_f1"]


## 1. Prediction target and available data

The classes `below / typical / above` describe performance **relative to the
post's own author**. For example, 80 reactions is strong performance for an author
who usually receives 20, but weak performance for one who usually receives 300.
The author's level is the median `log(1 + reactions)` over their earlier own
posts. The current post's residual relative to this level is compared with the
20th and 80th percentiles of training residuals; test thresholds stay unchanged.

`posts/` contains text, dates, content types, embedded posts, and engagement
snapshots; `authors/` contains author information, including follower counts;
`labels.csv` supplies targets and `split.json` defines the official author split.
The first five own posts are unlabelled but supply history. Videos and images
are not encoded: we use their structural metadata and text already present in JSON.


In [2]:
d = summary["dataset"]
display(pd.DataFrame([
    {"Split": "Train", "Authors": d["train_authors"], "Labelled posts": d["train_posts"]},
    {"Split": "Test", "Authors": d["test_authors"], "Labelled posts": d["test_posts"]},
]).set_index("Split"))


,Authors,Labelled posts
Split,,
Train,3295,71395
Test,824,17623


## 2. Evaluation protocol

The official test set is preserved: its authors do not appear in train. Stage one
used an author holdout inside train, seed 42: **56,847 posts / 2,636 authors** for
fitting and **14,548 / 659** for validation. Finalists and stage two were then
checked on three fixed author folds. OOF predictions are made for authors excluded
from the corresponding model's training data.

The primary metric is macro-F1, which gives all three classes equal weight; QWK
also reflects their ordering. TF-IDF, SVD, scaling, and category encoding are fitted
only on the fitting partition. Current reactions, residuals, and labels are excluded
from features. A validation/test author's own history may contain only earlier
posts. Past counters and follower counts are collection-time snapshots: this is
a benchmark, not a strict simulation of information available at publication time.


## 3. Starting point: the baseline

`baseline.py` uses scikit-learn's `HistGradientBoostingClassifier`: up to 300
iterations, learning rate 0.08, and balanced class weights. Numerical features
are scaled, and the content type is encoded with OneHotEncoder.

The baseline has **16 numerical features**: ten describe text and publication time
(length, lines, links, hashtags, mentions, questions, empty text, hour, and weekday);
four describe the author's level, historical spread, posting gap, and followers;
two describe the post's age at collection and the size of its preceding history.
Content type is a separate categorical feature. The model sees the shape of a post,
but not its words. The reproduced baseline achieved **0.5403 on test**; its E0
control achieved **0.5568** on the common validation split.


## 4. Recent history, writing style, and mentions

First, we tested whether an author's overall level misses recent changes.
The deviation of the last five posts' median improved E0 from **0.5568 to 0.5684**.
Adding a ten-post window and comparing consecutive groups of five reduced the
score to **0.5658**. Adding an unusual-gap feature to that version raised it to
**0.5708**. These results therefore must not be described as three successive gains.

Style was tested separately on top of E0: uppercase share, emoji, exclamation marks,
repeated letters, attachment counts and available text lengths, and deviations in length/links from the author's
usual behaviour gave **0.5877**. A separate block of mentions and their novelty gave
**0.5725**. Combining all blocks in E2 reached **0.6073**; this is a joint effect,
not the sum of their individual gains.

The result is **45 numerical features**: 16 original + 5 recent-history/rhythm +
15 style/attachment + 9 mention/repost features. Content type remains a separate
categorical feature.


In [3]:
comparisons = [
    ("Last 5 posts", "E1a_recent5", "E0_baseline"),
    ("Extend history to 10", "E1b_recent10", "E1a_recent5"),
    ("Unusual posting gap", "E1c_rhythm", "E1b_recent10"),
    ("Style", "E2a_style", "E0_baseline"),
    ("Mentions", "E2b_mentions", "E0_baseline"),
    ("All tabular features", "E2_all", "E0_baseline"),
]
display(pd.DataFrame([{"Addition": title, "Reference": base,
    "Reference macro-F1": round(f1(base), 4), "Candidate macro-F1": round(f1(candidate), 4),
    "Delta, pp": round(100 * (f1(candidate) - f1(base)), 3)}
    for title, candidate, base in comparisons]))


,Addition,Reference,Reference macro-F1,Candidate macro-F1,"Delta, pp"
0,Last 5 posts,E0_baseline,0.5568,0.5684,1.161
1,Extend history to 10,E1a_recent5,0.5684,0.5658,-0.261
2,Unusual posting gap,E1b_recent10,0.5658,0.5708,0.502
3,Style,E0_baseline,0.5568,0.5877,3.087
4,Mentions,E0_baseline,0.5568,0.5725,1.571
5,All tabular features,E0_baseline,0.5568,0.6073,5.052


## 5. Turning text into numbers

**TF-IDF** uses words and bigrams, up to 60,000 features, `min_df=3`, and logarithmic
term frequency. Enriched text combines the original post, quoted post, titles,
and available document transcripts. For boosting, the sparse matrix is compressed
to **128 TruncatedSVD coordinates**, fitted on the fitting partition.

**multilingual-e5-small** is frozen: gradients from our labels do not update it.
With the `query: ` prefix, it produces **384 normalized coordinates**. The 512-token
limit includes the prefix and special tokens: 256 tokens for the original text,
160 for quoted text, 64 for the post's own titles, and the remainder for its document.
Unused budget flows to subsequent sources. Embedded repost titles/transcripts are
available to TF-IDF but are not included in this E5 input.

Original text alone with logistic regression scored **0.4475**; enriching it gave
**0.4638**. Text alone was weaker than E0. Adding SVD to E2 gave **0.6247**, and
adding E5 gave **0.6250**: content is useful together with author context.


In [4]:
text_comparisons = [
    ("Enriched text, logistic regression", "E3b_text_enriched", "E3a_text_main"),
    ("E2 + 128 SVD, HGB", "E3c_svd_tabular", "E2_all"),
    ("E2 + 384 E5, HGB", "E4a_e5_hgb", "E2_all"),
    ("HGB instead of logistic regression on E2 + E5", "E4a_e5_hgb", "E4a_e5_linear"),
]
display(pd.DataFrame([{"Comparison": title, "Reference macro-F1": round(f1(base), 4),
    "Candidate macro-F1": round(f1(candidate), 4), "Delta, pp": round(100*(f1(candidate)-f1(base)), 3)}
    for title, candidate, base in text_comparisons]))


,Comparison,Reference macro-F1,Candidate macro-F1,"Delta, pp"
0,"Enriched text, logistic regression",0.4475,0.4638,1.639
1,"E2 + 128 SVD, HGB",0.6073,0.6247,1.739
2,"E2 + 384 E5, HGB",0.6073,0.6250,1.765
3,HGB instead of logistic regression on E2 + E5,0.5944,0.6250,3.059


## 6. Semantic author history and the V1 result

We tested whether topic typicality helps: similarity to the author's historical
profile, most recent post, and most similar earlier posts. These scalar features
were **added to E2 instead of the complete text vector**. SVD-history scored
**0.6066**, compared with E2's 0.6073; E5-history scored **0.6053**. Adding the
performance of three similar predecessors improved these to **0.6115** and
**0.6169**, but directly using the text vectors remained stronger.

Across three folds, a mixture of 25% E2 + 75% of the SVD model achieved OOF
**0.621574**. The corresponding E5 mixture achieved **0.624701** and became V1.
E5's additional gain was only **0.313 percentage points**: the overall improvement
cannot be attributed to the neural encoder alone. This OOF comparison is distinct
from the preceding holdout experiments and from the official test evaluation.


## 7. V2: similar posts from other authors

We compared **A = E2**, **B = E2 + E5**, and **C = E2 + TF-IDF/SVD**. The hypothesis
was that similar content may already have appeared among other authors' posts.
Neighbours are retrieved by cosine similarity of **E5 vectors**, including KNN
for family C; C's 128 SVD coordinates remain a separate text representation.

We take up to **20 distinct other fitting authors**: first their closest eligible
post, then the best authors. Posts must strictly precede the query. Both all prior
history and a window of up to 180 days were tested. The nine features are three
class fractions, mean residual, mean/maximum cosine, mean age, neighbour count,
and a missingness flag. Class fractions and residuals use `max(cosine, 0)` weights;
age and mean cosine are unweighted.

To prevent training-row features from revealing their own targets, we use inner
author splits: two parts within an outer fitting partition and three for final
training. The validation/test neighbour bank contains fitting authors only.
Neither future posts nor held-out authors populate this bank.


In [5]:
gates = {r["experiment"]: r for r in summary["feature_gates"]}
selected_blocks = [("A", "knn180"), ("B", "none"), ("C", "knn180")]
rows = []
for family, block in selected_blocks:
    record = gates[f"{family}/knn180"]
    base = record["macro_f1"] - record["pooled_delta"]
    score = base if block == "none" else record["macro_f1"]
    rows.append({"Family": family, "Selected block": block,
                 "Without added block": round(base, 4), "With selected block": round(score, 4),
                 "Delta, pp": round(100*(score-base), 3)})
display(pd.DataFrame(rows))


,Family,Selected block,Without added block,With selected block,"Delta, pp"
0,A,knn180,0.6019,0.6112,0.932
1,B,none,0.6228,0.6228,0.000
2,C,knn180,0.6194,0.6218,0.236


## 8. Changes that did not pass selection

A block was accepted if pooled macro-F1 increased by at least 0.002, at least two
folds improved, and no fold lost more than 0.005. A retained the 180-day window:
**+0.932 pp**, compared with **+0.846** for all history. C gained **+0.236 pp**;
its 95% author interval of [-0.003; +0.499] pp includes zero. Passing the selection
rule is therefore not equivalent to convincing statistical evidence. B retained no KNN.

Global organization features described popularity and co-mentions in earlier posts
from fitting authors, without reactions or labels. Changes for A/B/C were
**-0.029 / +0.073 / -0.021 pp**, so this block was rejected. None of the six org + KNN
combinations sufficiently improved on the selected individual block either.
This does not remove local organization novelty from the 45 tabular features,
and it does not assume that a friendship graph is available.


## 9. Boosting settings and blending

With the selected blocks, we tested 12 HGB configurations: the original; 15/63
leaves; minimum leaf sizes of 50/100; L2=1/10; no class balancing; learning rate
0.04 with up to 600 iterations; and three combinations of tree size, leaf size,
and L2. This produced **36 configurations and 15 A/B/C mixtures** with a weight step of 0.25.

The selected mixture was **50% B/leaves63 + 50% C/knn180/combo63**. Both branches
use balanced weights, learning rate 0.08, up to 300 iterations, and automatic early
stopping. B uses up to 63 leaves, minimum leaf size 20, L2=0; C uses up to 63 leaves, minimum
leaf size 100, L2=1. Dimensions are **45+384=429** numerical features for B and
**45+128+9=182** for C, plus content type.

The mixture achieved OOF **0.638197**, compared with **0.623742** for the V1 recipe
recomputed in the same run: **+1.445 pp**. This combines feature, tuning, and blending
effects. The historical 0.624701 is not substituted for the current control;
the reason for their difference has not been established.


In [6]:
tuning = pd.DataFrame(summary["family_tuning"])[["family", "block", "param", "macro_f1"]]
display(tuning.rename(columns={"family": "Family", "block": "Block", "param": "Setting", "macro_f1": "OOF macro-F1"}).round(4))
validation = pd.DataFrame(summary["validation"])
display(validation.iloc[:2][["variant", "macro_f1"]].rename(columns={"variant": "Mixture", "macro_f1": "OOF macro-F1"}).round(4))


,Family,Block,Setting,OOF macro-F1
0,A,knn180,leaves63,0.6156
1,B,none,leaves63,0.6295
2,C,knn180,combo63,0.6299


,Mixture,OOF macro-F1
0,V1 recipe control,0.6237
1,50% E5 + 50% TF-IDF/SVD + KNN180,0.6382


## 10. From probabilities to the final class

The next hypothesis was that the argmax of learned probabilities need not be the
best decision rule for macro-F1. We tested 81 pairs of below/above multipliers from
0.8 to 1.2 in steps of 0.05, keeping typical=1. Within each outer validation fold,
authors are split in half (seed 42 + fold number): weights are selected on one half
and evaluated on the other, then the roles are reversed. The base model was trained
on neither half.

Cross-calibration of **six separately fitted rules** achieved **0.648045**, +0.985 pp over V2.
A single set of weights, `[0.8, 1, 0.8]`, was then fitted on all OOF rows. Its
**0.647751 on those same rows is not a validation score**. The grid was not expanded
after selecting a boundary point. The final class is
`argmax(0.8*p_below, p_typical, 0.8*p_above)`; these weighted scores are not presented
as new normalized probabilities.


In [7]:
decision = summary["decision_validation"]
validation_scores = {r["stage"]: r["macro_f1"] for r in summary["validation"]}
display(pd.DataFrame([
    {"Evaluation": "V2: original decision", "Macro-F1": validation_scores["V2"]},
    {"Evaluation": "Cross-calibration of six rules", "Macro-F1": validation_scores["Decision rule"]},
    {"Evaluation": "Single rule fitted on all OOF: NOT validation", "Macro-F1": decision["full_oof_refit_macro_f1"]},
]).round(6))
ci = decision["paired_author_bootstrap"]
print(f"Procedure gain: {100*ci['observed_delta']:.3f} pp; 95% CI [{100*ci['ci_low']:.3f}; {100*ci['ci_high']:.3f}] pp")


,Evaluation,Macro-F1
0,V2: original decision,0.638197
1,Cross-calibration of six rules,0.648045
2,Single rule fitted on all OOF: NOT validation,0.647751


Procedure gain: 0.985 pp; 95% CI [0.786; 1.193] pp


## 11. Official test results and errors

After fixing the recipe, the models were trained on all **71,395 training posts**.
Final macro-F1 **0.639033** exceeds the baseline by **9.870 pp** and V2 by **1.433 pp**.
Most of the last step's gain comes from `typical`: its F1 increased **0.6788→0.7223**;
`above` increased **0.6561→0.6586**, while `below` decreased **0.5393→0.5362**.

For `below`, final precision is approximately 0.505 and recall 0.572. **41.0%**
of true below posts were classified as typical; the extreme below→above error
is less frequent. The decision rule reduced false extreme predictions on typical
posts, but did not solve the recognition of weak posts. Macro-F1 alone is therefore
insufficient to explain how the model's behaviour changed.


In [8]:
names = {"baseline": "Baseline", "v1": "V1", "v2": "V2", "final": "V2 + decision rule"}
test = pd.DataFrame.from_dict(summary["test_metrics"], orient="index")
display(test[["macro_f1", "qwk", "accuracy"]].rename(index=names, columns={"macro_f1": "Macro-F1", "qwk": "QWK", "accuracy": "Accuracy"}).round(4))
cm = pd.DataFrame(summary["test_metrics"]["final"]["confusion_matrix"], index=d["classes"], columns=d["classes"])
errors = pd.DataFrame(index=d["classes"])
errors["F1 V2"] = summary["test_metrics"]["v2"]["per_class_f1"]
errors["Final F1"] = summary["test_metrics"]["final"]["per_class_f1"]
errors["Final precision"] = [cm.loc[c, c] / cm[c].sum() for c in d["classes"]]
errors["Final recall"] = [cm.loc[c, c] / cm.loc[c].sum() for c in d["classes"]]
display(errors.round(4))
print(f"True below → typical: {100*cm.loc['below', 'typical']/cm.loc['below'].sum():.2f}%")
final = summary["final"]
print(f"Gain over V2: {100*final['delta_macro_f1_vs_v2']:.3f} pp; 95% author CI [{100*final['ci_low']:.3f}; {100*final['ci_high']:.3f}] pp")


,Macro-F1,QWK,Accuracy
Baseline,0.5403,0.4603,0.5445
V1,0.6142,0.5583,0.6251
V2,0.6247,0.5658,0.6402
V2 + decision rule,0.6390,0.5741,0.6692


,F1 V2,Final F1,Final precision,Final recall
below,0.5393,0.5362,0.5049,0.5716
typical,0.6788,0.7223,0.7438,0.7020
above,0.6561,0.6586,0.6479,0.6697


True below → typical: 41.01%
Gain over V2: 1.433 pp; 95% author CI [1.035; 1.862] pp


## 12. Implementation and reproduction

`kairos/experiment_features.py` builds tabular features and history;
`kairos/experiment_text.py` handles TF-IDF/SVD; `kairos/experiment_encode.py` handles E5;
`kairos/experiment_knn.py` and `kairos/experiment_org.py` implement the additional blocks.
`kairos.experiment` runs V1; `kairos/experiment_stage2.py` compares and fits V2.
The final mixture is fixed in `configs/solution_v2.json`, and the decision rule
in `configs/solution_v2_decision.json` and `calibrate_decision.py`.

After installing dependencies and placing the core dataset in `data/`:

```bash
python -m kairos.label
python calibrate_decision.py --config configs/solution_v2_decision.json
python evaluate.py --predictions results/v2_decision/final/predictions.csv
```

With a validated E5 cache, training runs on CPU without Torch. Without that cache,
the encoder defaults to CUDA and requires PyTorch and `requirements-gpu.txt`.
`--action apply` reruns only the decision rule on existing V2 probabilities.


## 13. Limits of the conclusions

Models were selected using train, but V1/V2 test results were already known during
later steps: the final result belongs to a sequential experiment, not a new
independent confirmation. Repeated OOF use for selection also introduces optimism.
The author bootstrap (1,000 resamples) describes fixed predictions and does not
account for uncertainty from training and the entire selection process.

Reported metrics were obtained on Linux CPU/Python 3.12.3. The full command also
runs on Mac/Python 3.10, but its predictions differ; cross-platform bitwise agreement
is not guaranteed. The results support combining author context, text, and a tuned
decision rule. They do not demonstrate a universal isolated benefit from a neural
encoder, organization features, or KNN.
